# 图片 → Gaussian Splat → Three.js 查看器（Colab 版）

在 Colab GPU 上启动 threed 项目，得到一个可从本地浏览器打开的链接（需 Chrome/Edge 113+ 支持 WebGPU）。

按顺序运行下面的单元格；全部幂等，重复运行不会重复 clone 或重新下载权重。

In [ ]:
# 1. 获取项目代码并安装前端依赖
import os

ROOT = '/content/threed'
if not os.path.isdir(ROOT):
    !git clone https://github.com/novvoo/threed.git /content/threed
%cd {ROOT}
!npm install

In [ ]:
# 2. 准备 TripoSplat 后端：克隆仓库 + venv + 依赖
import os

ts = os.path.join(os.getcwd(), 'TripoSplat')
if not os.path.isdir(ts):
    !git clone --depth 1 https://github.com/VAST-AI-Research/TripoSplat.git {ts}

# venv 只作为插件的 python 入口（--system-site-packages 复用系统包，包括 Colab 预装的 torch）；
# Colab 的 venv 缺 ensurepip，因此 --without-pip 创建，依赖装到系统 Python
venv = os.path.join(ts, '.venv')
if not os.path.exists(os.path.join(venv, 'bin', 'python')):
    !python3 -m venv --without-pip --system-site-packages {venv}
!pip install -q numpy safetensors pillow tqdm huggingface_hub

In [ ]:
# 3. 下载模型权重（约 4.2GB，首次较慢；已有则跳过）
import os

marker = os.path.join(os.getcwd(), 'TripoSplat', 'ckpts', 'diffusion_models', 'triposplat_fp16.safetensors')
if not os.path.exists(marker):
    # 新版 CLI 叫 hf，旧版叫 huggingface-cli，二者兜底
    !hf download VAST-AI/TripoSplat --local-dir TripoSplat/ckpts || huggingface-cli download VAST-AI/TripoSplat --local-dir TripoSplat/ckpts
else:
    print('权重已存在，跳过下载')

In [ ]:
# 4. 启动 dev server（后台）并输出可从本地浏览器打开的链接
import os, socket, subprocess, time

# 已在运行则先停掉，保证可重复执行
subprocess.run(['bash', '-c', "pkill -f 'vite' || true"])

log = open('/content/vite.log', 'w')
subprocess.Popen(['npm', 'run', 'dev', '--', '--host', '0.0.0.0', '--strictPort'],
                 stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    time.sleep(1)
    try:
        socket.create_connection(('127.0.0.1', 5180), timeout=1).close()
        break
    except OSError:
        pass
else:
    print('启动失败，日志：')
    print(open('/content/vite.log').read())

try:
    from google.colab.output import eval_js
    url = eval_js('google.colab.kernel.proxyPort(5180)')
    print(f'\n✅ 在本地浏览器（Chrome/Edge 113+，需支持 WebGPU）打开：{url}')
except Exception:
    print('\n✅ Colab 虚拟机内访问：http://localhost:5180')
print('服务器日志：/content/vite.log（转换任务的 python 输出也会出现在这里）')